**Timeline**

* Попытка обучить на своих чеках opensource-модель (не хватило GPU)
* Прицепляем ChatGPT для обработки после Tesseract
* fuzzywuzzy для поиска в БД



In [ ]:
!apt-get install tesseract-ocr -y
!apt-get install tesseract-ocr-rus -y
!pip install pytesseract pandas python-telegram-bot==20.7 opencv-python-headless psycopg2-binary fuzzywuzzy python-Levenshtein
!pip install openai --upgrade -q

print("Библиотеки установлены")

In [ ]:
import os, re, asyncio
import numpy as np
import io, logging
from PIL import Image, ImageEnhance
import pytesseract, cv2, psycopg2
from psycopg2 import pool
from google.colab import userdata
from fuzzywuzzy import fuzz, process
from openai import OpenAI

logging.basicConfig(level=logging.ERROR)
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'

OPENAI_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_KEY)

print("Клиент инициализирован")

In [ ]:
DB_HOST = "eco-scan-ecobot.e.aivencloud.com"
DB_PORT = "10408"
DB_NAME = "defaultdb"
DB_USER = "avnadmin"
DB_PASS = userdata.get('DB_PASSWORD')

db_Pool = psycopg2.pool.SimpleConnectionPool(1, 10, host=DB_HOST, port=DB_PORT, database=DB_NAME, user=DB_USER, password=DB_PASS, sslmode='require')
print("Подключено к БД")

def load():
    conn = db_Pool.getconn()
    try:
        with conn.cursor() as cursor:
            cursor.execute("""SELECT name, matched_product_id FROM lenta WHERE name IS NOT NULL
                UNION ALL SELECT name, matched_product_id FROM magnit WHERE name IS NOT NULL
                UNION ALL SELECT name, matched_product_id FROM okey WHERE name IS NOT NULL
                UNION ALL SELECT name, matched_product_id FROM perekrestok WHERE name IS NOT NULL
                UNION ALL SELECT name, matched_product_id FROM x5 WHERE name IS NOT NULL""")
            store_prods = {}
            for name, matched_id in cursor.fetchall():
                if name and len(name) > 3:
                    store_prods[name.lower()] = matched_id

            cursor.execute("""SELECT pr.id, pr.name, c.name as category, pr.carbon_footprint FROM product_reference pr JOIN categories c ON pr.category_id = c.id""")
            products = {}
            for product_id, name, category, footprint in cursor.fetchall():
                products[product_id] = {'name': name, 'category': category, 'co2': float(footprint) if footprint else 0.0}
            print(f"Загружено названий из магазинов: {len(store_prods)}")
            print(f"Загружено продуктов из справочника: {len(products)}")
            return store_prods, products
    except:
        print("Ошибка")
        return {}, {}
    finally:
        db_Pool.putconn(conn)

store_prods, prod_info = load()

In [ ]:
def preproc_img(image_bytes):
    try:
        nparr = np.frombuffer(image_bytes, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        if img is None:
            return None

        height, width = img.shape[:2]
        if width < 1500:
            scale = 2000 / width
            new_width = int(width * scale)
            new_height = int(height * scale)
            img = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_CUBIC)

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced = clahe.apply(gray)
        denoised = cv2.fastNlMeansDenoising(enhanced, h=30)
        binary = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)

        pil_img = Image.fromarray(binary)
        sharpen = ImageEnhance.Sharpness(pil_img)
        shrpnd = sharpen.enhance(1.5)
        return shrpnd
    except:
        print("Ошибка предобработки")
        return None

In [ ]:
async def gpt(ocr_txt):
    if not OPENAI_KEY:
        print("Ключ OpenAI API не настроен")
        return ocr_txt

    try:
        prompt = f"""Ты эксперт по исправлению OCR-ошибок в чеках. ВАЖНЫЕ ПРАВИЛА:
1. Текст чеков на РУССКОМ языке. Не переводи ничего на английский или другие языки.
2. ТОРГОВЫЕ НАИМЕНОВАНИЯ (бренды, названия продуктов) НЕЛЬЗЯ заменять или переводить.
3. Исправляй только явные опечатки (буквы, цифры).
4. НЕ удаляй пробелы и дефисы в названиях.
5. Сохраняй оригинальную структуру строк.
6. Не переводи и не переименовывай продукты.
7. Сохраняй КАЖДУЮ строку чека.
8. НЕ УДАЛЯЙ и НЕ ОБЪЕДИНЯЙ строки с продуктами.

Оригинальный OCR-текст:
{ocr_txt}

Верни ТОЛЬКО исправленный текст, сохраняя все строки, без пояснений и комментариев."""

        response = client.chat.completions.create(model="gpt-4.1-mini", messages=[{"role": "user", "content": prompt}], max_tokens=2000, temperature=0.1)

        fxd_txt = response.choices[0].message.content
        return fxd_txt
    except:
        print("Ошибка")
        return ocr_txt

In [ ]:
async def img_txt(photo_file):
    try:
        file = await photo_file.get_file()
        ph_bytes = await file.download_as_bytearray()
        processed = preproc_img(ph_bytes)
        if processed:
            text1 = pytesseract.image_to_string(processed, lang='rus+eng', config='--oem 3 --psm 6')
            text2 = pytesseract.image_to_string(processed, lang='rus+eng', config='--oem 3 --psm 3')
            text3 = pytesseract.image_to_string(processed, lang='rus+eng', config='--oem 3 --psm 11')
            combined = text1 + "\n" + text2 + "\n" + text3

            combined = await gpt(combined)

            return combined.lower()
        return ""
    except:
        print("Ошибка")
        return ""

https://pypi.org/project/pytesseract/

https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html

https://pillow.readthedocs.io/en/stable/reference/ImageEnhance.html

https://platform.openai.com/docs/api-reference

https://github.com/seatgeek/fuzzywuzzy

https://github.com/ztane/python-Levenshtein/